# Interactive Inference: Qwen2.5-VL + DisasterM3 LoRA
Use this notebook to quickly test your fine-tuned model on individual images.

In [2]:
!pip install -q -U torch torchvision torchaudio transformers accelerate qwen-vl-utils pillow

In [4]:
!pip install -q -U torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:0000:0100:01


In [5]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from PIL import Image
from IPython.display import display

print("Loading Merged DisasterM3 model from Hugging Face...")

# Load your custom merged model natively!
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP",
    device_map="auto",
    torch_dtype=torch.float16
)
processor = AutoProcessor.from_pretrained("AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP")

print("Model ready for showcasing!")


Loading Merged DisasterM3 model from Hugging Face...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Model ready for showcasing!


In [8]:
from qwen_vl_utils import process_vision_info

def ask_model(image_path, question):
    # 1. Visually display the image in the notebook
    print("\nInput Image:")
    try:
        img = Image.open(image_path).convert("RGB")
        img.thumbnail((500, 500)) # Resize for nice notebook display
        display(img)
    except Exception as e:
        print(f"[Could not load image preview: {e}]")

    # 2. Prepare the prompt
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    # 3. Process and Generate
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    print("\n🧠 Thinking...")
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    return processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]


In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os

# 1. Grab a list of images from the dataset to populate the dropdown
image_dir = "/kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/test_images/"
# Find all PNGs and take the first 100 for the dropdown menu
image_files = glob.glob(os.path.join(image_dir, "*.png"))[:100] 
options_dict = {os.path.basename(f): f for f in image_files}

# 2. Create the UI Widgets
image_dropdown = widgets.Dropdown(
    options=options_dict,
    description='Select Image:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

question_input = widgets.Text(
    value='Which of the following disaster types does this image show? A: Earthquake, B: Fire, C: Flood, D: Hurricane.',
    description='Question:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

run_button = widgets.Button(
    description=' Analyze Image',
    button_style='primary',
    icon='search'
)

output_display = widgets.Output()

# 3. Define what happens when the button is clicked
def on_run_clicked(b):
    with output_display:
        clear_output(wait=True) # Clears the previous result
        image_path = image_dropdown.value
        question = question_input.value
        
        print(f"Question: {question}")
        
        # Call the model (from Cell 3)
        answer = ask_model(image_path, question)
        
        print("====================================")
        print(f"DisasterM3 Answer: {answer}")
        print("====================================")

run_button.on_click(on_run_clicked)

# 4. Display the interactive form!
ui = widgets.VBox([image_dropdown, question_input, run_button, output_display])
display(ui)
